In [204]:
!pip install rdkit
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from rdkit import Chem
from rdkit.Chem import Descriptors
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
seed = 42
print(train.shape)
print(test.shape)
train

(320, 7)
(597, 7)


,Title,CC50_uM,IC50_uM_VACV,SI VACV,SMILES,strain,Culture
0,PAV-825,15.0,2.18,6.881,CCC(CC)C1=CC2=C(C=C1)N=C1C(C(C)(C)C)=CC(N(CC)C...,"Vaccinia virus Western Reserve, HeLa cells",HeLa cells
1,11c,26.2,NaN,10000.000,C[C@@]12CC[C@H](C2(C)C)C[C@H]1OC(=O)c3ccc(Br)cc3,Vaccinia virus (strain Copenhagen). culture Ve...,Vero
2,10a,217.2,NaN,10000.000,C1C[C@@H](C2)C(C)(C)[C@@H]([C@]12C)NC(=O)c3ccccc3,Vaccinia virus (strain Copenhagen). culture Ve...,Vero
3,sid103692656,100.0,100.00,1.000,O=C(c1ccccc1Br)n1ccnc1-c1ccc(Cl)cc1,"Vaccinia virus, HFF cells",Human foreskin fibroblast
4,sid160666886,45.0,NaN,10000.000,CC(C(=O)NNC(=O)NC1CCCCC1)c1cccc(Cc2ccccc2)c1,"Vaccinia virus, HELA cells",HELA cells
...,...,...,...,...,...,...,...
315,7j-as-526,104.5,6.80,15.368,CC1(C)[C@@H]2CC[C@@]1(C)[C@H](NC(=O)C1=CN=CC=C...,Vaccinia virus (strain Copenhagen). culture Ve...,Vero
316,11,258.0,38.70,6.667,CC1(C)[C@@H]2CC[C@@]1(C)/C(=N/NC(=O)C1=CC=CO1)C2,Vaccinia virus (strain Copenhagen). culture Vero,Vero
317,sid134450510,303.0,NaN,10000.000,Cc1cc(C)cc(C(=O)c2cc(Cl)ccc2OCCn2c(C)cc(=O)[nH...,Vaccinia virus,MT-4
318,sid312354110,100.0,23.00,4.348,Cc1nc2cc(C(F)(F)F)ccc2n1C[C@@H]1CCCN2CCCC[C@H]12,"Vaccinia virus, Vero cells",Vero


In [205]:
train['MolWt'] = train['SMILES'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
train['LogP'] = train['SMILES'].apply(lambda x: Descriptors.MolLogP(Chem.MolFromSmiles(x)))
train['NumAtoms'] = train['SMILES'].apply(lambda x: Chem.MolFromSmiles(x).GetNumAtoms())
train['RingCount'] = train['SMILES'].apply(lambda x: Descriptors.RingCount(Chem.MolFromSmiles(x)))
train['NumHDonors'] = train['SMILES'].apply(lambda x: Descriptors.NumHDonors(Chem.MolFromSmiles(x)))
train['NumHAcceptors'] = train['SMILES'].apply(lambda x: Descriptors.NumHAcceptors(Chem.MolFromSmiles(x)))
train['TPSA'] = train['SMILES'].apply(lambda x: Descriptors.TPSA(Chem.MolFromSmiles(x)))

test['MolWt'] = test['SMILES'].apply(lambda x: Descriptors.MolWt(Chem.MolFromSmiles(x)))
test['LogP'] = test['SMILES'].apply(lambda x: Descriptors.MolLogP(Chem.MolFromSmiles(x)))
test['NumAtoms'] = test['SMILES'].apply(lambda x: Chem.MolFromSmiles(x).GetNumAtoms())
test['RingCount'] = test['SMILES'].apply(lambda x: Descriptors.RingCount(Chem.MolFromSmiles(x)))
test['NumHDonors'] = test['SMILES'].apply(lambda x: Descriptors.NumHDonors(Chem.MolFromSmiles(x)))
test['NumHAcceptors'] = test['SMILES'].apply(lambda x: Descriptors.NumHAcceptors(Chem.MolFromSmiles(x)))
test['TPSA'] = test['SMILES'].apply(lambda x: Descriptors.TPSA(Chem.MolFromSmiles(x)))


train.drop(['SMILES', 'CC50_uM', 'IC50_uM_VACV'], axis=1, inplace=True)
test.drop(['SMILES', 'CC50_uM', 'IC50_uM_VACV'], axis=1, inplace=True)
train.describe()

,SI VACV,MolWt,LogP,NumAtoms,RingCount,NumHDonors,NumHAcceptors,TPSA
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2744.650250,365.475000,3.187677,25.321875,3.500000,1.378125,4.384375,76.309781
std,4706.256961,112.834332,2.362305,7.678053,1.543517,1.321687,2.877718,46.959248
min,0.000000,150.141000,-3.684600,11.000000,1.000000,0.000000,1.000000,16.130000
25%,2.000000,295.116250,2.037900,20.000000,2.000000,1.000000,3.000000,41.377500
50%,20.260500,339.892000,3.402730,24.000000,3.000000,1.000000,4.000000,70.060000
75%,10000.000000,400.443500,4.659480,28.000000,4.000000,2.000000,5.000000,92.782500
max,36066.667000,1101.243000,12.977100,77.000000,9.000000,11.000000,22.000000,356.810000


In [206]:
title = 'Title'
target = 'SI VACV'
cat_cols = ['strain', 'Culture']
num_cols = [c for c in train.columns if c not in [title, target] + cat_cols]

In [207]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
        ('num', make_pipeline(
            SimpleImputer(strategy='median'),
            StandardScaler()
        ), num_cols
        )
    ],
    verbose_feature_names_out=False
    )

train_SI_na_mean = train['SI VACV'].fillna(train['SI VACV'].mean())
train_drop_SI_na = train.drop(['SI VACV'], axis=1)

X_train = train.drop(columns=[title])
X_test = test.drop(columns=['index', title])

train_prep = pd.DataFrame(preprocessor.fit_transform(X_train), columns=preprocessor.get_feature_names_out())
test_prep = pd.DataFrame(preprocessor.transform(X_test), columns=preprocessor.get_feature_names_out())

train_prep['SI VACV'] = train['SI VACV'].values
train_prep['SI VACV']

,SI VACV
0,6.881
1,10000.000
2,10000.000
3,1.000
4,10000.000
...,...
315,15.368
316,6.667
317,10000.000
318,4.348


In [208]:
from sklearn.model_selection import train_test_split

X = train_prep[[c for c in train_prep.columns if c not in [target, title]]]
y = train_prep[target]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=seed, shuffle=True
)

In [209]:
model = DummyRegressor()
model.fit(X_train, y_train)
y_pred = model.predict(X_valid)
mse = mean_squared_error(y_valid, y_pred)
print(mse)

18412781.36395638


In [210]:
model.fit(X, y)
y_pred = model.predict(test_prep)

In [213]:
submission = pd.DataFrame({
    'index': test_prep.index,
    target: y_pred
})

submission.to_csv('submission.csv', index=False)